# Sample test runs: image → caption with a ClipCap prototype

This notebook is the first **end-to-end test of the proposed model**. It trains a small ClipCap-style captioner on the phase 1 outputs and captions **5 unseen Karpathy test images**.

```
image ─► CLIP ViT-B/32 (frozen) ─► 512-d embedding ─► mapping MLP ─► 10 prefix vectors ─► GPT-2 ─► caption
```

* **Mapping network:** an MLP that turns one CLIP embedding into 10 vectors in GPT-2's input-embedding space (a "prefix").
* **Decoder:** GPT-2 small (124M), fine-tuned together with the MLP, as in the MLP variant of ClipCap (Mokady et al., 2021).
* **Training data:** the 14,907 cleaned train captions. Val loss picks the best epoch.

This is a **prototype**: a short run on a 3k-image training set. It shows that the pipeline works end to end; it is not the final model.

**Prerequisite:** the files from `Image_Captioning_Baseline.ipynb` (`outputs/` and `data/`).

**On Google Colab:** *Runtime → Change runtime type → T4 GPU*. Put `colab_bundle.zip` (the phase 1 files) in `MyDrive/image-captioning/` on Google Drive, then *Runtime → Run all*. The notebook mounts Drive, unpacks the bundle once and saves its results back to Drive.

In [ ]:
import os, sys, math, time, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2LMHeadModel, GPT2TokenizerFast, CLIPModel, CLIPProcessor, get_linear_schedule_with_warmup

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip -q install "transformers>=4.44"
    from google.colab import drive
    drive.mount("/content/drive")
PROJECT_DIR = Path("/content/drive/MyDrive/image-captioning") if IN_COLAB else Path.cwd()
DATA_DIR, OUT_DIR = PROJECT_DIR / "data" / "flickr30k_subset", PROJECT_DIR / "outputs"
IMG_DIR = DATA_DIR / "images"
EMB_PATH, TOK_PATH = OUT_DIR / "clip_image_embeddings.pt", OUT_DIR / "gpt2_tokens.pt"
CKPT_DIR = OUT_DIR / "clipcap_prototype"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

SEED = 42
PREFIX_LEN = 10
EPOCHS = 5
SMOKE_STEPS = int(os.environ.get("SMOKE_STEPS", 0))   # >0 only for a quick test of the code
BATCH_SIZE = 32
LR_MAPPING, LR_GPT2 = 1e-4, 3e-5
WARMUP_STEPS = 200

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
# On Colab: unpack the phase 1 files once, if you uploaded colab_bundle.zip to the project folder on Drive
BUNDLE = PROJECT_DIR / "colab_bundle.zip"
if BUNDLE.exists() and not EMB_PATH.exists():
    import zipfile
    zipfile.ZipFile(BUNDLE).extractall(PROJECT_DIR)
    print("unpacked", BUNDLE)
assert EMB_PATH.exists() and TOK_PATH.exists(), "run Image_Captioning_Baseline.ipynb first (or upload colab_bundle.zip)"
print("device:", DEVICE)

## 1. Data: phase 1 outputs as (embedding, tokens) pairs

In [ ]:
emb_file, tok_file = torch.load(EMB_PATH), torch.load(TOK_PATH)
IMAGE_EMB = emb_file["embeddings"]                         # [5000, 512], L2-normalised
IMAGE_IDS = emb_file["image_ids"]
captions_df = pd.read_csv(DATA_DIR / "captions_clean.csv", dtype={"image_id": str})


class CaptionEmbeddingDataset(Dataset):
    def __init__(self, split):
        keep = [i for i, s in enumerate(tok_file["split"]) if s == split]
        self.input_ids = tok_file["input_ids"][keep]
        self.attention_mask = tok_file["attention_mask"][keep]
        self.image_index = tok_file["image_index"][keep]

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, i):
        labels = self.input_ids[i].clone()
        labels[self.attention_mask[i] == 0] = -100
        return {"clip_embedding": IMAGE_EMB[self.image_index[i]], "input_ids": self.input_ids[i],
                "attention_mask": self.attention_mask[i], "labels": labels}


train_ds, val_ds = CaptionEmbeddingDataset("train"), CaptionEmbeddingDataset("val")
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=64)
print(f"train pairs: {len(train_ds):,} | val pairs: {len(val_ds):,} | steps/epoch: {len(train_dl)}")

## 2. Model: CLIP prefix → GPT-2

The MLP maps the 512-d CLIP embedding to `PREFIX_LEN × 768` numbers, reshaped into 10 vectors of GPT-2's embedding size (768). GPT-2 reads the 10 prefix vectors followed by the caption tokens and is trained to predict each next caption token. Prefix positions get label `-100`, so the loss covers only the caption.

In [ ]:
class ClipCaptionModel(nn.Module):
    def __init__(self, prefix_len=PREFIX_LEN, clip_dim=512):
        super().__init__()
        self.gpt = GPT2LMHeadModel.from_pretrained("gpt2")
        d = self.gpt.config.n_embd                                   # 768
        self.prefix_len = prefix_len
        hidden = (clip_dim + d * prefix_len) // 2
        self.mapping = nn.Sequential(nn.Linear(clip_dim, hidden), nn.Tanh(), nn.Linear(hidden, d * prefix_len))

    def prefix(self, clip_embedding):
        return self.mapping(clip_embedding).view(-1, self.prefix_len, self.gpt.config.n_embd)

    def forward(self, clip_embedding, input_ids, attention_mask, labels):
        b = input_ids.size(0)
        prefix = self.prefix(clip_embedding)
        tokens = self.gpt.transformer.wte(input_ids)
        ones = torch.ones(b, self.prefix_len, dtype=attention_mask.dtype, device=input_ids.device)
        ignore = torch.full((b, self.prefix_len), -100, dtype=labels.dtype, device=input_ids.device)
        return self.gpt(inputs_embeds=torch.cat([prefix, tokens], 1),
                        attention_mask=torch.cat([ones, attention_mask], 1),
                        labels=torch.cat([ignore, labels], 1)).loss


model = ClipCaptionModel().to(DEVICE)
n_map = sum(p.numel() for p in model.mapping.parameters()) / 1e6
n_gpt = sum(p.numel() for p in model.gpt.parameters()) / 1e6
print(f"mapping MLP: {n_map:.1f}M params | GPT-2: {n_gpt:.0f}M params")

## 3. Training

AdamW with a higher learning rate for the new mapping network (`1e-4`) than for the pretrained GPT-2 (`3e-5`), and a linear warm-up then decay. After each epoch we measure the val loss and keep the weights of the best epoch.

In [ ]:
def to_device(batch):
    return {k: v.to(DEVICE) for k, v in batch.items()}


@torch.no_grad()
def evaluate(dl):
    model.eval()
    total, n = 0.0, 0
    for batch in dl:
        b = to_device(batch)
        total += model(**b).item() * len(b["input_ids"]); n += len(b["input_ids"])
    model.train()
    return total / n


optimizer = torch.optim.AdamW([{"params": model.mapping.parameters(), "lr": LR_MAPPING},
                               {"params": model.gpt.parameters(), "lr": LR_GPT2}], weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(optimizer, WARMUP_STEPS, EPOCHS * len(train_dl))

history, best_val, best_state = [], float("inf"), None
(OUT_DIR / "train_log.txt").write_text("")
model.train()
for epoch in range(1, EPOCHS + 1):
    t0, running = time.time(), 0.0
    for step, batch in enumerate(tqdm(train_dl, desc=f"epoch {epoch}", leave=False), 1):
        loss = model(**to_device(batch))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step(); optimizer.zero_grad()
        running += loss.item()
        if SMOKE_STEPS and step >= SMOKE_STEPS:
            break
    val_loss = evaluate(val_dl)
    history.append({"epoch": epoch, "train_loss": running / len(train_dl), "val_loss": val_loss})
    if val_loss < best_val:
        best_val = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    line = (f"epoch {epoch}: train {running / len(train_dl):.3f} | val {val_loss:.3f} "
            f"(perplexity {math.exp(val_loss):.1f}) | {(time.time() - t0) / 60:.1f} min")
    print(line)
    with open(OUT_DIR / "train_log.txt", "a") as f:   # progress you can check while the notebook runs
        f.write(line + "\n")

model.load_state_dict(best_state)
history = pd.DataFrame(history)
best_epoch = int(history.loc[history.val_loss.idxmin(), "epoch"])
print(f"best epoch: {best_epoch} (val loss {best_val:.3f})")

In [ ]:
plt.figure(figsize=(6, 3.5))
plt.plot(history.epoch, history.train_loss, "o-", label="train")
plt.plot(history.epoch, history.val_loss, "o-", label="val")
plt.xlabel("epoch"); plt.ylabel("cross-entropy loss"); plt.title("ClipCap prototype training")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

CKPT_DIR.mkdir(parents=True, exist_ok=True)
torch.save({"state_dict": best_state, "prefix_len": PREFIX_LEN, "history": history.to_dict("records")},
           CKPT_DIR / "clipcap_mlp_gpt2.pt")
print("saved", CKPT_DIR / "clipcap_mlp_gpt2.pt")

## 4. Caption generation

GPT-2 starts from the 10 prefix vectors and writes the caption token by token, using **beam search** (5 beams; repeated 3-grams blocked) until it emits `<|endoftext|>` or reaches 30 tokens.

In [ ]:
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")


@torch.no_grad()
def generate_caption(clip_embedding, num_beams=5, max_new_tokens=30):
    model.eval()
    prefix = model.prefix(clip_embedding.to(DEVICE).unsqueeze(0))
    out = model.gpt.generate(inputs_embeds=prefix,
                             attention_mask=torch.ones(prefix.shape[:2], dtype=torch.long, device=DEVICE),
                             max_new_tokens=max_new_tokens, num_beams=num_beams, no_repeat_ngram_size=3,
                             eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0], skip_special_tokens=True).strip()

## 5. Five sample test runs

Five images drawn at random (seed 42) from the **Karpathy test split**. The model never saw these images or their captions during training. Each row shows the generated caption beside two of the five human reference captions.

As a quick quality check we also compute the **CLIP similarity** between the image and each caption (cosine of CLIP embeddings, higher = closer match). A generated caption scoring close to the human references means it describes the image about as well as a person did.

In [ ]:
is_test = torch.tensor([s == "test" for s in tok_file["split"]])
test_ids = sorted({IMAGE_IDS[i] for i in tok_file["image_index"][is_test].tolist()})
sample_ids = random.Random(SEED).sample(test_ids, 5)
emb_row = {iid: i for i, iid in enumerate(IMAGE_IDS)}

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE).eval()
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")


@torch.no_grad()
def clip_similarity(image_emb, texts):
    inputs = clip_processor(text=texts, return_tensors="pt", padding=True, truncation=True).to(DEVICE)
    t = clip_model.get_text_features(**inputs)
    t = t if torch.is_tensor(t) else t.pooler_output
    return (torch.nn.functional.normalize(t, dim=-1).cpu() @ image_emb).tolist()


runs = []
for iid in sample_ids:
    e = IMAGE_EMB[emb_row[iid]]
    refs = captions_df[captions_df.image_id == iid].caption.tolist()
    gen = generate_caption(e)
    sims = clip_similarity(e, [gen] + refs)
    runs.append({"image_id": iid, "generated": gen, "reference_1": refs[0], "reference_2": refs[1],
                 "clip_sim_generated": round(sims[0], 3), "clip_sim_refs_mean": round(float(np.mean(sims[1:])), 3)})

runs = pd.DataFrame(runs)
runs.to_csv(OUT_DIR / "sample_test_runs.csv", index=False)
pd.set_option("display.max_colwidth", 80)
runs

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 6.2))
for ax, r in zip(axes, runs.itertuples()):
    ax.imshow(Image.open(IMG_DIR / f"{r.image_id}.jpg"))
    ax.axis("off")
    ax.set_title(f"Generated: {r.generated}\n\nReference: {r.reference_1}", fontsize=9, wrap=True, loc="left")
plt.tight_layout()
fig.savefig(OUT_DIR / "sample_test_runs.png", dpi=110, bbox_inches="tight")
plt.show()

print(f"mean CLIP similarity | generated: {runs.clip_sim_generated.mean():.3f} | "
      f"human references: {runs.clip_sim_refs_mean.mean():.3f}")

## 6. Takeaways

* The full pipeline works end to end: **image → CLIP embedding → mapping MLP → GPT-2 → caption**.
* Even after a short training run on 3k images, the generated captions name the main subject and action of each test image.
* Next steps for the final model: train on more of the 29k-image Karpathy train split, compare an MLP with a transformer mapping network, try a frozen vs. fine-tuned GPT-2, and score the full 1,000-image test set with BLEU-4, METEOR, CIDEr and CLIPScore.